In [3]:
# import  os
# num_cores = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = num_cores
# os.environ["OMP_NUM_THREADS"] = num_cores
# os.environ["MKL_NUM_THREADS"] = num_cores

In [1]:
import sys
sys.path.append('../')

import numpy as np
import scipy.sparse as ssp
import matplotlib.pyplot as plt
import qutip as qt
import scqubits as scq
from matplotlib.colors import LogNorm
from tqdm import tqdm
from qutip.qip.operations import rz, cz_gate
import cmath
from sympy import symbols
from scipy.sparse.linalg import eigsh
import utils_2Q_gate_zp as ut
from joblib import Parallel, delayed
from IPython.display import display, Math
# ut.set_fig_font() ### Set various sizes in plotting
import networkx as nx
from multiprocessing import Pool
import pandas as pd

In [ ]:
truc1, truc_tot, charge_pick  = 300, 1000, True
normalize = True
folder = f'../../data/3ncut_two_zeropi/truc1={truc1}_truc2={truc_tot}_pick={charge_pick}/'
eval_tot = pd.read_csv(folder+ 'eval_tot.txt').to_numpy().flatten()
n_theta0_dress = 2*np.pi* np.load(folder+'n_theta0_dress.npy')
n_theta1_dress = 2*np.pi* np.load(folder+'n_theta1_dress.npy')
hspace_full = pd.read_csv(folder+ 'hspace_full.txt').to_numpy().flatten().tolist()

truc_tot_2 = 1000
truc_list = np.arange(truc_tot_2)
hspace_full = hspace_full[:truc_tot_2]
eval_tot = eval_tot[:truc_tot_2]
n_theta0_dress = ut.truncate_2(n_theta0_dress, truc_list)
n_theta1_dress = ut.truncate_2(n_theta1_dress, truc_list)
hspace_logi = ['0-0', '0-2', '2-0', '2-2']

In [ ]:
W_20_50 = eval_tot[hspace_full.index('5-0')] - eval_tot[hspace_full.index('2-0')]
wd = W_20_50 
core_states = hspace_logi + ['5-0']
print('len(eval_tot)=', len(eval_tot))
print('len(index_state)=', len(hspace_full))
# print('index_state=', index_state)

len(eval_tot)= 1000
len(index_state)= 1000


### Truncation Estimate

In [22]:
drive_term = n_theta1_dress
A = 0.02
######
# max_n_ij = np.max(drive_term) # Normalize based on n
population_rate = np.zeros((truc_tot, truc_tot), dtype=np.complex128)
population_rate_log = np.zeros((truc_tot, truc_tot), dtype=np.complex128)
rabi_df = []
G = nx.DiGraph()
for i, s_i in enumerate(hspace_full):
    for j, s_j in enumerate(hspace_full):
        if i < j:
            n_ij = drive_term[i, j]
            # normalization =  n_ij/max_n_ij
            delta = abs(wd - (eval_tot[j] - eval_tot[i]))
            population_rate[i,j] = ((A*n_ij)**2) / ((A*n_ij)**2 + delta**2)
            if population_rate[i,j] > 0:
                population_rate_log[i, j] = -np.log(population_rate[i,j])
                G.add_edge(s_i, s_j, weight=population_rate_log[i, j]) # Construct the graph

    rabi_df.append({"order": i, "i": s_i})
rabi_df = pd.DataFrame(rabi_df)
rabi_df.index = rabi_df["i"]
print('shape(population_rate_log)=', np.shape(population_rate_log))
rabi_df

shape(population_rate_log)= (1000, 1000)


,order,i
i,,
0-0,0,0-0
0-1,1,0-1
1-0,2,1-0
0-2,3,0-2
2-0,4,2-0
...,...,...
22-45,995,22-45
2-109,996,2-109
79-8,997,79-8


In [23]:
if truc_tot < 30:
    nx.draw(G, pos=nx.shell_layout(G), with_labels=True)
print(G['0-0']['1-0'])
qt.Qobj(population_rate_log)

{'weight': (3.792474984859608-0j)}


Quantum object: dims = [[1000], [1000]], shape = (1000, 1000), type = oper, isherm = False
Qobj data =
[[ 0.          3.39446926  3.79247498 ... 45.53747301  0.
   0.        ]
 [ 0.          0.          0.         ...  0.         48.25155394
  49.41486501]
 [ 0.          0.          0.         ...  0.         43.39082362
  42.40390702]
 ...
 [ 0.          0.          0.         ...  0.         13.3353803
  12.41530211]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]]

In [24]:
def shortest_path_to_core(target):
    shortest_path = ""
    shortest_path_len = np.inf
    for source in core_states[:-1]:
        if nx.has_path(G, source, target):
            path = nx.shortest_path(G, source=source, target=target,
                                    weight="weight")
            path_len = nx.shortest_path_length(G, source=source, target=target,
                                                weight="weight")
        if path_len < shortest_path_len:
            shortest_path_len = path_len
            shortest_path = ",".join([str(x) for x in path])
    return target, (shortest_path_len, shortest_path)

cutoff = 2
def all_path_to_core(target):
    path_tot = []
    if target in core_states:
        weight_tot = 1
    else:
        weight_tot = 0
        for source in core_states:
            for path in nx.all_simple_paths(G, source, target, cutoff=cutoff):
                weight_tot += np.exp( - nx.path_weight(G, path,'weight') )
                path_tot.append(path)
    return target, (weight_tot, path_tot)
print(shortest_path_to_core('1-0'))
print(all_path_to_core('0-4'))
print(all_path_to_core('0-1'))

('1-0', ((3.792474984859608+0j), '0-0,1-0'))
('0-4', ((0.0006271105136581368+0j), [['0-0', '0-1', '0-4'], ['0-0', '1-0', '0-4']]))
('0-1', ((0.03355836019110092+0j), [['0-0', '0-1']]))


In [25]:
target = '0-4'
source = '0-0'
paths = nx.all_simple_paths(G, source=source, target=target, cutoff=cutoff)
weight_tot = sum([np.exp(- nx.path_weight(G, path=path, weight='weight'))
                   for path in paths])
print(weight_tot)
for path in nx.all_simple_paths(G, source=source, target=target, cutoff=cutoff):
    print(path)

(0.0006271105136581368+0j)
['0-0', '0-1', '0-4']
['0-0', '1-0', '0-4']


In [26]:
# Find shortest path for each node
pool = Pool(processes=150)
shortest_path = {}
rabi_df["path"] = ""
rabi_df["path_len"] = 1.0
for idx, path in tqdm(pool.imap_unordered(shortest_path_to_core, hspace_full),
                total=truc_tot):
    shortest_path[idx[0]] = path
    rabi_df.at[idx, "path"] = path[1]
    rabi_df.at[idx, "path_len"] = np.exp(-path[0])

df_short = rabi_df.sort_values("path_len", ascending=False)
print('shortest_path:')
df_short.iloc[:30]

100%|██████████| 1000/1000 [00:03<00:00, 254.50it/s]

shortest_path:


,order,i,path,path_len
i,,,,
0-0,0,0-0,0-0,1.000000+0.000000j
2-2,13,2-2,2-2,1.000000+0.000000j
0-2,3,0-2,0-2,1.000000+0.000000j
2-0,4,2-0,2-0,1.000000+0.000000j
5-0,10,5-0,"2-0,5-0",1.000000-0.000000j
5-2,27,5-2,"2-2,5-2",0.328100-0.000000j
5-1,21,5-1,"2-0,5-0,5-1",0.063630-0.000000j
0-1,1,0-1,"0-0,0-1",0.033558-0.000000j
2-1,9,2-1,"2-0,2-1",0.027706-0.000000j


In [27]:
df_short_200 = df_short.iloc[:200].sort_values("order", ascending=True)
print('state_short :')
data = df_short_200['i']
for i in range(0, len(data), dim):  # Step size of 10
    print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

state_short :
'0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1' ,
'5-0', '1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '0-9', '2-4', '9-0' ,
'1-5', '5-1', '4-2', '0-12', '2-5', '1-8', '12-0', '5-2', '0-13', '0-16' ,
'8-1', '4-4', '13-0', '15-0', '2-8', '1-9', '9-1', '8-2', '5-4', '4-5' ,
'0-18', '2-9', '1-12', '0-20', '0-21', '18-0', '9-2', '12-1', '5-5', '0-24' ,
'4-8', '1-13', '20-0', '8-4', '1-16', '22-0', '2-12', '13-1', '0-25', '15-1' ,
'0-26', '5-8', '4-9', '12-2', '2-13', '9-4', '2-16', '8-5', '25-0', '13-2' ,
'1-18', '15-2', '0-28', '5-9', '4-12', '0-30', '18-1', '12-4', '9-5', '1-20' ,
'1-21', '0-33', '8-8', '0-34', '1-24', '2-18', '0-36', '4-13', '22-1', '4-16' ,
'5-12', '15-4', '24-1', '33-0', '2-20', '18-2', '2-21', '8-9', '1-25', '9-8' ,
'0-39', '12-5', '2-24', '5-13', '20-2', '5-16', '13-5', '25-1', '0-42', '15-5' ,
'4-18', '2-25', '39-0', '8-12', '2-26', '9-9', '18-4', '12-8', '1-30', '4-20' ,
'0-45', '4-21', '1-33', '8-16', '22-4', '4-24', '5-18', '1-34',

In [28]:
# Find all_path_to_core
pool = Pool(processes=150)
shortest_path = {}
rabi_df["path"] = ""
rabi_df["path_len"] = 1.0
for idx, path in tqdm(pool.imap_unordered(all_path_to_core, hspace_full),
                total=truc_tot):
    shortest_path[idx[0]] = path
    rabi_df.at[idx, "path"] = path[1]
    rabi_df.at[idx, "path_len"] = path[0]

df_all = rabi_df.sort_values("path_len", ascending=False)
print('all_path -- cutoff=', cutoff)
df_all.iloc[:30]

100%|██████████| 1000/1000 [00:01<00:00, 855.94it/s]

all_path -- cutoff= 2


,order,i,path,path_len
i,,,,
0-0,0,0-0,[],1.000000+0.000000j
0-2,3,0-2,[],1.000000+0.000000j
2-0,4,2-0,[],1.000000+0.000000j
2-2,13,2-2,[],1.000000+0.000000j
5-0,10,5-0,[],1.000000+0.000000j
5-2,27,5-2,"[[0-0, 5-2], [0-2, 5-2], [2-0, 5-2], [2-2, 5-2...",0.328100+0.000000j
5-1,21,5-1,"[[0-0, 0-1, 5-1], [0-0, 1-0, 5-1], [0-0, 0-5, ...",0.127326+0.000000j
0-1,1,0-1,"[[0-0, 0-1]]",0.033558+0.000000j
2-1,9,2-1,"[[0-0, 2-1], [0-2, 2-1], [2-0, 2-1]]",0.027706+0.000000j


In [29]:
df_all_200 = df_all.iloc[:200].sort_values("order", ascending=True)
print('state_all :')
data = df_all_200['i']
for i in range(0, len(data), dim):  # Step size of 10
    print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

state_all :
'0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1' ,
'5-0', '1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '0-9', '2-4', '9-0' ,
'1-5', '5-1', '4-2', '0-12', '2-5', '1-8', '12-0', '5-2', '0-13', '0-16' ,
'8-1', '4-4', '13-0', '15-0', '2-8', '1-9', '9-1', '8-2', '5-4', '4-5' ,
'0-18', '2-9', '1-12', '0-20', '0-21', '18-0', '9-2', '12-1', '5-5', '0-24' ,
'4-8', '1-13', '20-0', '8-4', '1-16', '22-0', '2-12', '13-1', '24-0', '0-25' ,
'15-1', '0-26', '5-8', '4-9', '12-2', '2-13', '9-4', '2-16', '8-5', '25-0' ,
'13-2', '1-18', '15-2', '5-9', '4-12', '0-30', '18-1', '12-4', '9-5', '1-20' ,
'1-21', '0-33', '8-8', '0-34', '0-35', '1-24', '2-18', '0-36', '4-16', '5-12' ,
'15-4', '24-1', '2-20', '18-2', '2-21', '8-9', '1-25', '9-8', '1-26', '0-39' ,
'12-5', '2-24', '5-13', '5-16', '22-2', '0-42', '15-5', '2-25', '8-12', '2-26' ,
'9-9', '0-44', '1-30', '0-45', '0-46', '1-33', '22-4', '4-24', '5-18', '1-34' ,
'15-8', '2-28', '9-12', '24-4', '12-9', '18-5', '2-30', '0-52', '

In [ ]:
=

In [30]:
aa = df_all_200['i'].to_numpy().tolist()
index = [hspace_full.index(i) for i in aa]
index

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 72,
 73,
 75,
 76,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 94,
 97,
 98,
 99,
 101,
 102,
 104,
 105,
 107,
 108,
 109,
 110,
 111,
 112,
 114,
 116,
 117,
 120,
 123,
 125,
 127,
 128,
 129,
 133,
 136,
 138,
 140,
 143,
 147,
 148,
 150,
 151,
 153,
 156,
 157,
 158,
 160,
 164,
 165,
 166,
 167,
 170,
 172,
 173,
 175,
 177,
 178,
 179,
 181,
 182,
 183,
 185,
 187,
 193,
 199,
 204,
 208,
 209,
 210,
 227,
 236,
 243,
 254,
 257,
 260,
 263,
 265,
 269,
 271,
 274,
 276,
 278,
 284,
 285,
 290,
 295,
 303,
 304,
 310,
 324,
 327,
 329,
 332,
 333,
 348,
 364,
 366,
 379,
 387,
 391,
 392,
 401,
 402,
 403,
 406,
 409

In [31]:
df_all_200 = df_all.iloc[:200].sort_values("order", ascending=True)
print('state_all :')
data = df_all_200['i']
for i in range(0, len(data), dim):  # Step size of 10
    print(", ".join(f"'{x}'" for x in data[i:i + dim]), ',')

state_all :
'0-0', '0-1', '1-0', '0-2', '2-0', '0-4', '4-0', '1-1', '0-5', '2-1' ,
'5-0', '1-2', '0-8', '2-2', '8-0', '1-4', '4-1', '0-9', '2-4', '9-0' ,
'1-5', '5-1', '4-2', '0-12', '2-5', '1-8', '12-0', '5-2', '0-13', '0-16' ,
'8-1', '4-4', '13-0', '15-0', '2-8', '1-9', '9-1', '8-2', '5-4', '4-5' ,
'0-18', '2-9', '1-12', '0-20', '0-21', '18-0', '9-2', '12-1', '5-5', '0-24' ,
'4-8', '1-13', '20-0', '8-4', '1-16', '22-0', '2-12', '13-1', '24-0', '0-25' ,
'15-1', '0-26', '5-8', '4-9', '12-2', '2-13', '9-4', '2-16', '8-5', '25-0' ,
'13-2', '1-18', '15-2', '5-9', '4-12', '0-30', '18-1', '12-4', '9-5', '1-20' ,
'1-21', '0-33', '8-8', '0-34', '0-35', '1-24', '2-18', '0-36', '4-16', '5-12' ,
'15-4', '24-1', '2-20', '18-2', '2-21', '8-9', '1-25', '9-8', '1-26', '0-39' ,
'12-5', '2-24', '5-13', '5-16', '22-2', '0-42', '15-5', '2-25', '8-12', '2-26' ,
'9-9', '0-44', '1-30', '0-45', '0-46', '1-33', '22-4', '4-24', '5-18', '1-34' ,
'15-8', '2-28', '9-12', '24-4', '12-9', '18-5', '2-30', '0-52', '

In [32]:
list1 = df_short_200['i']
list2 = df_all_200['i']
common_elements = [item for item in list1 if item in list2]
only_in_list1 = [item for item in list1 if item not in list2]
only_in_list2 = [item for item in list2 if item not in list1]
unique_elements = only_in_list1 + only_in_list2

# print("Common elements:", common_elements)
print("Only in state_short:", len(only_in_list1), only_in_list1)
print("Only in state_all:", len(only_in_list2), only_in_list2)
# print("Unique elements:", unique_elements)


Only in state_short: 48 ['0-28', '4-13', '22-1', '33-0', '20-2', '13-5', '25-1', '4-18', '39-0', '18-4', '12-8', '4-20', '4-21', '8-16', '1-36', '33-1', '46-0', '25-4', '15-9', '8-18', '12-12', '18-8', '54-0', '13-12', '8-21', '12-16', '20-8', '4-30', '12-13', '22-8', '25-5', '15-12', '9-18', '18-9', '13-16', '15-13', '44-1', '41-2', '35-4', '9-21', '46-1', '4-46', '12-25', '24-13', '41-5', '30-9', '13-25', '18-18']
Only in state_all: 48 ['24-0', '0-35', '1-26', '22-2', '0-44', '0-46', '0-53', '1-39', '9-16', '0-54', '0-57', '1-45', '9-20', '2-44', '15-16', '9-24', '0-68', '5-36', '0-73', '2-53', '9-26', '0-78', '18-16', '4-44', '2-54', '5-42', '0-83', '2-60', '5-44', '8-39', '9-33', '9-34', '9-35', '5-46', '2-64', '5-52', '5-53', '2-66', '2-68', '5-55', '2-73', '2-76', '5-59', '9-44', '5-65', '2-83', '5-68', '5-83']


In [33]:
print('cutoff=', cutoff)
target = '5-2'
print(shortest_path_to_core(target))
idx, path = all_path_to_core(target)
for i in path[1]:
    print(i)


cutoff= 2
('5-2', ((1.1144374429708779+0j), '2-2,5-2'))
['0-0', '5-2']
['0-2', '5-2']
['2-0', '5-2']
['2-2', '5-2']
['5-0', '2-2', '5-2']
['5-0', '0-9', '5-2']
['5-0', '2-4', '5-2']
['5-0', '9-0', '5-2']
['5-0', '1-5', '5-2']
['5-0', '5-1', '5-2']
['5-0', '4-2', '5-2']
['5-0', '0-12', '5-2']
['5-0', '1-8', '5-2']
['5-0', '12-0', '5-2']


In [34]:
print('cutoff=', cutoff)
target = '5-2'
print(shortest_path_to_core(target))
idx, path = all_path_to_core(target)
for i in path[1]:
    print(i)


cutoff= 2
('5-2', ((1.1144374429708779+0j), '2-2,5-2'))
['0-0', '5-2']
['0-2', '5-2']
['2-0', '5-2']
['2-2', '5-2']
['5-0', '2-2', '5-2']
['5-0', '0-9', '5-2']
['5-0', '2-4', '5-2']
['5-0', '9-0', '5-2']
['5-0', '1-5', '5-2']
['5-0', '5-1', '5-2']
['5-0', '4-2', '5-2']
['5-0', '0-12', '5-2']
['5-0', '1-8', '5-2']
['5-0', '12-0', '5-2']


In [35]:
# Find shortest path for each node
pool = Pool(processes=150)
shortest_path = {}
rabi_df["path"] = ""
rabi_df["path_len"] = 0.0
rabi_df["degree"] = -1
for idx, path in tqdm(pool.imap_unordered(shortest_path_to_core, hspace_full),
                total=truc_tot):
    shortest_path[idx[0]] = path
    rabi_df.at[idx, "path"] = path[1]
    rabi_df.at[idx, "path_len"] = np.exp(-path[0])
    rabi_df.at[idx, "degree"] = path[1].count(",")

# print(population_rate[index_state.index('0-0'), index_state.index('0-1')] *
#       population_rate[index_state.index('0-1'), index_state.index('0-4')])
# print(population_rate[index_state.index('0-0'), index_state.index('1-0')] *
#       population_rate[index_state.index('1-0'), index_state.index('0-4')])
print(rabi_df.sort_values("path_len", ascending=False).iloc[:15])
# qt.Qobj(population_rate[:10,:10])

100%|██████████| 1000/1000 [00:03<00:00, 258.22it/s]

     order    i         path            path_len  degree
i                                                       
0-0      0  0-0          0-0  1.000000+0.000000j       0
2-2     13  2-2          2-2  1.000000+0.000000j       0
0-2      3  0-2          0-2  1.000000+0.000000j       0
2-0      4  2-0          2-0  1.000000+0.000000j       0
5-0     10  5-0      2-0,5-0  1.000000-0.000000j       1
5-2     27  5-2      2-2,5-2  0.328100-0.000000j       1
5-1     21  5-1  2-0,5-0,5-1  0.063630-0.000000j       2
0-1      1  0-1      0-0,0-1  0.033558-0.000000j       1
2-1      9  2-1      2-0,2-1  0.027706-0.000000j       1
1-0      2  1-0      0-0,1-0  0.022540-0.000000j       1
0-5      8  0-5      0-2,0-5  0.014840-0.000000j       1
2-5     24  2-5      2-2,2-5  0.013964-0.000000j       1
1-2     11  1-2      0-2,1-2  0.013392-0.000000j       1
4-0      6  4-0  0-0,0-1,4-0  0.007998-0.000000j       2
5-5     48  5-5  2-2,5-2,5-5  0.005663-0.000000j       2
